In [1]:
print("heloo")

heloo


In [2]:
## Arxiv--Research
## Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

C:\Users\asrit\AppData\Local\Temp\ipykernel_34580\554445524.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun


In [3]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [4]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [5]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

api_wrapper_arxiv = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

result = arxiv.invoke("1706.03762")
print(result)

Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c


In [6]:
tools=[wiki,arxiv]

In [7]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [9]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,embeddings)
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001CAB5807AD0>, search_kwargs={})

In [10]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name

'langsmith-search'

In [11]:
tools=[wiki,arxiv,retriever_tool]

In [12]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\asrit\\Videos\\langchain-projects\\langchain-agent-search-engine\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001CAB58209A0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001CAFDBD13A0>)]

In [13]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [14]:
from langchain.chat_models import init_chat_model
model=init_chat_model(model="groq:openai/gpt-oss-120b")

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI search assistant.
    Use the available tools to answer user questions accurately.
    If you don't know the answer, say so."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


In [16]:
from langchain_classic.agents import create_tool_calling_agent

agent = create_tool_calling_agent(model, tools, prompt)

agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageCh

In [17]:
## Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[l

In [18]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


root cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application

{'input': 'Tell me about Langsmith',
 'output': '**LangSmith – an observability & evaluation platform for LLM‑powered applications**\n\nLangSmith (hosted at\u202f`smith.langchain.com`) is the monitoring, debugging, and evaluation suite built by the LangChain team for anyone who runs production‑grade language‑model (LLM) workloads—agents, chatbots, Retrieval‑Augmented Generation (RAG) pipelines, etc. It gives you “full visibility” into what your LLM app does in the wild, lets you spot and fix problems quickly, and helps you improve quality over time.\n\n---\n\n### Core capabilities\n\n| Area | What LangSmith provides | Why it matters |\n|------|------------------------|----------------|\n| **Tracing** | Every request (a *trace*) records the full sequence of LLM calls, tool invocations, prompts, responses, and metadata. | Re‑create exactly what happened in production, down to each turn of an agent. |\n| **Observability dashboards** | Real‑time metrics, performance charts, error rates, to

In [19]:
agent_executor.invoke({"input":"what is machine learning "})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks wi**Machine learning (ML)** is a sub‑field of artificial intelligence that focuses on designing and studying algorithms that can **learn patterns from data** and then **generalize those patterns to make predictions or decisions on new, unseen data**. Rather than being explicitly programmed for every possible situation, a machine‑learning system improves its performance automatically as it is exposed to more data.

### Core ideas
| Concept | Explanation |
|---------|-------------|
| **Learning from data** | Algorithms ingest a dataset (examples, features, labels) and infer a model that captures underlying relationships. |
| **Generalization** 

{'input': 'what is machine learning ',
 'output': '**Machine learning (ML)** is a sub‑field of artificial intelligence that focuses on designing and studying algorithms that can **learn patterns from data** and then **generalize those patterns to make predictions or decisions on new, unseen data**. Rather than being explicitly programmed for every possible situation, a machine‑learning system improves its performance automatically as it is exposed to more data.\n\n### Core ideas\n| Concept | Explanation |\n|---------|-------------|\n| **Learning from data** | Algorithms ingest a dataset (examples, features, labels) and infer a model that captures underlying relationships. |\n| **Generalization** | The model should perform well not only on the training data but also on new inputs it has never seen. |\n| **Statistical inference** | Most ML methods rely on probability and statistics to estimate model parameters and assess uncertainty. |\n| **Automation of tasks** | Once trained, the model

In [20]:
agent_executor.invoke({"input":"what the paper  1706.03762 about"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `arxiv` with `{'query': 'arXiv:1706.03762'}`


Published: 2020-02-12
Title: GLU Variants Improve Transformer
Authors: Noam Shazeer
Summary: Gated Linear Units (arXiv:1612.08083) consist of the component-wise product of two linear projections, one of which is first passed through a sigmoid functio
Invoking: `wikipedia` with `{'query': 'Attention Is All You Need'}`


Page: Attention Is All You Need
Summary: "Attention Is All You Need" is a 2017 research paper on machine learning authored by eight scientists and engineers working at Google. The paper introduced a new deep learning architecture known as the transfo**Paper:** arXiv:1706.03762 –

{'input': 'what the paper  1706.03762 about',
 'output': '**Paper:**\u202farXiv:1706.03762 – *“Attention Is All You Need”*  \n**Authors:**\u202fAshish\u202fVaswani, Noam\u202fShazeer, Niki\u202fParmar, Jakob\u202fUszkoreit, Llion\u202fJones, Aidan\u202fN.\u202fGomez, Łukasz\u202fKaiser, Illia\u202fPolosukhin (Google Brain)  \n**Year:**\u202f2017 (published on arXiv\u202fJune\u202f2017, later presented at NIPS\u202f2017)\n\n---\n\n### Core Idea\nThe paper proposes a **new neural network architecture for sequence‑to‑sequence tasks called the *Transformer***. Unlike earlier models (RNNs, LSTMs, GRUs, or convolutional seq2seq models) that rely on recurrence or convolutions to capture positional dependencies, the Transformer **uses only attention mechanisms** to relate different positions in a sequence.\n\n---\n\n### Key Contributions\n\n| Contribution | What it is | Why it matters |\n|--------------|------------|----------------|\n| **Self‑Attention (Scaled Dot‑Product Attention)** | Compu